## Simple Calculator Tracing using Phoenix
Here we can use the Python SDK to develop the simple calculator agent, then save the agent to a config.yaml and run it from there.

In [ ]:
import os
import sys

# Import the NeMo-Agent-Toolkit module
module_path = os.path.abspath('../../../src/')
if module_path not in sys.path:
    sys.path.insert(0, module_path)

In [ ]:
from nat.front_ends.fastapi.fastapi_front_end_config import FastApiFrontEndConfig
from nat.llm.nim_llm import NIMModelConfig
from nat.observability.register import ConsoleLoggingMethodConfig
from nat.observability.register import FileLoggingMethod
from nat.observability.register import console_logging_method
from nat.observability.register import file_logging_method
from nat.plugins.phoenix.register import PhoenixTelemetryExporter
from nat.plugins.phoenix.register import phoenix_telemetry_exporter
from nat.tool.datetime_tools import CurrentTimeToolConfig
from nat.tool.datetime_tools import current_datetime
from nat.utils.sdk.nat_agent import NatReactAgent
from nat.utils.sdk.nat_front_end import NatFrontEnd
from nat.utils.sdk.nat_general_configuraton import NatGeneralConfiguration
from nat.utils.sdk.nat_llm import NatLLM
from nat.utils.sdk.nat_logger import NatLogger
from nat.utils.sdk.nat_tool import NatTool
from nat.utils.sdk.nat_tool_group import NatToolGroup
from nat.utils.sdk.nat_tracer import NatTracer
from nat_simple_calculator.register import CalculatorToolConfig
from nat_simple_calculator.register import calculator

llm = NatLLM(config=NIMModelConfig(model_name="nvdev/meta/llama-3.1-70b-instruct", temperature=0.0, max_tokens=1024), )

current_time_tool = NatTool(
    config=CurrentTimeToolConfig(),
    function=current_datetime,
    name="current_datetime",
)
calculator_tool_group = NatToolGroup(
    config=CalculatorToolConfig(),
    tool_group=calculator,
    name="calculator",
)

# Define the configuration
console_logger = NatLogger(
    config=ConsoleLoggingMethodConfig(level="WARN", ),
    function=console_logging_method,
    name="console_logger",
)
file_logger = NatLogger(
    config=FileLoggingMethod(
        path="./.tmp/nat_simple_calculator.log",
        level="DEBUG",
        create_if_not_exists=True,
    ),
    function=file_logging_method,
    name="file_logger",
)
phoenix_tracer = NatTracer(
    config=PhoenixTelemetryExporter(endpoint="http://localhost:6006/v1/traces", project="simple_calculator"),
    function=phoenix_telemetry_exporter,
    name="phoenix_tracer",
)
get_time_endpoint = FastApiFrontEndConfig.Endpoint(path="/get_time",
                                                   method="POST",
                                                   description="Gets the current time",
                                                   function_name=current_time_tool.tool_name)
front_end_configuration = NatFrontEnd(endpoints=[get_time_endpoint],
                                      cors=FastApiFrontEndConfig.CrossOriginResourceSharing(allow_origins=["*"]))

general_agent_configuration = NatGeneralConfiguration(loggers=[console_logger, file_logger],
                                                      tracers=[phoenix_tracer],
                                                      front_end_configuration=front_end_configuration)

agent = NatReactAgent(
    configuration=general_agent_configuration,
    tools=[current_time_tool],
    tool_groups=[calculator_tool_group],
    llm=llm,
    verbose=True,
    parse_agent_response_max_retries=3,
)

/Users/spastoriza/Documents/Programming/public/nat-fork/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
import os
from pathlib import Path

path_to_yaml = Path(os.getcwd(), "config", "config-phoenix.yaml").resolve()

# Create the config directory if it doesn't exist
if not path_to_yaml.parent.exists():
    os.makedirs(path_to_yaml.parent)

# Save the agent to a config file
agent.save_to_config_file(path_to_yaml)

# Print out the config file content
with open(path_to_yaml) as f:
    print(f.read())

None of PyTorch, TensorFlow >= 2.0, or Flax have been found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.


general:
  telemetry:
    logging:
      console_logger:
        _type: console
        level: WARN
      file_logger:
        _type: file
        path: ./.tmp/nat_simple_calculator.log
        level: DEBUG
        create_if_not_exists: true
    tracing:
      phoenix_tracer:
        _type: phoenix
        project: simple_calculator
        endpoint: http://localhost:6006/v1/traces
  front_end:
    _type: fastapi
    endpoints:
    - method: POST
      description: Gets the current time
      path: /get_time
      function_name: current_datetime
    cors:
      allow_origins:
      - '*'

functions:
  current_datetime:
    _type: current_datetime

function_groups:
  calculator:
    _type: calculator

llms:
  nim:
    _type: nim
    model: nvdev/meta/llama-3.1-70b-instruct
    max_tokens: 1024
    temperature: 0.0

workflow:
  _type: react_agent
  llm_name: nim
  verbose: true
  tool_names:
  - current_datetime
  - calculator
  parse_agent_response_max_retries: 3



In [4]:
await agent.prompt('What is 4 * 50 plus the current hour?')

'220.0'